# 04 Modeling: Multi-Year Earnings Prediction

This notebook builds and evaluates the earnings prediction models.


## What this notebook does

- trains the 1-year, 4-year, and 5-year models
- compares actual and predicted earnings
- creates the saved multi-year results used in the app

This notebook contains the main modeling work for the final project.


In [3]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

root = Path("__file__").resolve().parents[1]
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

WindowsPath('C:/Users/sebas/PycharmProjects/Git')

In [4]:
def collect(state: str = "FL", per_page: int = 100) -> pd.DataFrame:
    fields = ",".join([
        "id",
        "school.name",

        "location.lat",
        "location.lon",

        "latest.programs.cip_4_digit.code",
        "latest.programs.cip_4_digit.unit_id",
        "latest.programs.cip_4_digit.title",
        "latest.programs.cip_4_digit.school.type",
        "latest.programs.cip_4_digit.credential.level",
        "latest.programs.cip_4_digit.distance",

        "latest.school.locale",
        "latest.school.carnegie_size_setting",
        "latest.admissions.admission_rate.overall",
        "latest.student.demographics.median_family_income",
        "latest.student.students_with_pell_grant",
        "latest.school.open_admissions_policy",
        "latest.student.demographics.age_entry",
        "latest.school.title_iv.eligibility_type",
        "latest.programs.cip_4_digit.earnings.1_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.1_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.2_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.2_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.3_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.3_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.4_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.4_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.5_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.5_yr.working_not_enrolled.overall_count"
    ])

    params = {
        "api_key": SCORECARD_KEY,
        "school.state": state,
        "fields": fields,
        "per_page": str(per_page),

        # only programs with earnings reported
        "latest.programs.cip_4_digit.earnings.4_yr.overall_median_earnings__range": "1.."
    }

    dfs = []

    params["page"] = "0"
    response = get_with_retries(BASE_URL, params=params, timeout=30)
    data = get_json_or_raise(response)

    total = int(data["metadata"]["total"])
    per_page_actual = int(data["metadata"]["per_page"])
    total_pages = math.ceil(total / per_page_actual)

    META = [
        "id",
        "school.name",
        "location.lat",
        "location.lon",
        "latest.school.locale",
        "latest.school.carnegie_size_setting",
        "latest.admissions.admission_rate.overall",
        "latest.student.demographics.median_family_income",
        "latest.student.students_with_pell_grant",
        "latest.school.open_admissions_policy",
        "latest.student.demographics.age_entry",
        "latest.school.title_iv.eligibility_type",
    ]

    def page_to_df(data):
        results = [r for r in data.get("results", []) if r.get("latest.programs.cip_4_digit")]
        return pd.json_normalize(
            results,
            record_path=["latest.programs.cip_4_digit"],
            meta=META,
            errors="ignore",
        )

    dfs.append(page_to_df(data))

    for page in range(1, total_pages):
        params["page"] = str(page)
        response = get_with_retries(BASE_URL, params=params, timeout=30)
        data = get_json_or_raise(response)
        dfs.append(page_to_df(data))

    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    print(f"Total pages fetched: {total_pages}")
    print(f"Total Rows/Programs ingested: {len(df)}")
    return df

def get_json_or_raise(response: requests.Response):
    # Raise for HTTP errors early (4xx/5xx)
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        ct = response.headers.get("Content-Type", "")
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"HTTP {response.status_code} for {response.url}\n"
            f"Content-Type: {ct}\n"
            f"Body preview:\n{body_preview}"
        ) from e

    # Check content-type sanity (helps catch HTML responses)
    ct = response.headers.get("Content-Type", "")
    if "json" not in ct.lower():
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"Expected JSON but got Content-Type: {ct}\n"
            f"URL: {response.url}\n"
            f"Body preview:\n{body_preview}"
        )

    # Parse JSON with a clearer error if it fails
    try:
        return response.json()
    except json.JSONDecodeError as e:
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"JSON decode failed for {response.url}\n"
            f"Body preview:\n{body_preview}"
        ) from e
    
    import time
import random
import requests

def get_with_retries(url, params, tries=5, timeout=30):
    last = None
    for i in range(tries):
        r = requests.get(url, params=params, timeout=timeout, headers={"Accept": "application/json"})
        if r.status_code < 500:
            return r
        last = r
        time.sleep((2 ** i) + random.random())
    return last

# Removed features
---
### 1

```
"latest.student.demographics.avg_family_income"
"latest.student.demographics.median_hh_income"
```
overlaps with ```"latest.student.demographics.median_family_income"```

---
### 2

```
"latest.academics.program_reporter.programs_offered"
```
~83% is null

---
### 3

```
latest.admissions.test_requirements
```
~46% is null and overlaps with ```latest.admissions.admission_rate.overall  ```

---
### 4

```
"latest.admissions.sat_scores.50th_percentile.critical_reading",
"latest.admissions.sat_scores.50th_percentile.math",
"latest.admissions.act_scores.50th_percentile.cumulative",
"latest.admissions.act_scores.50th_percentile.english",
"latest.admissions.act_scores.50th_percentile.math",
"latest.admissions.sat_scores.average.overall",
"latest.admissions.act_scores.midpoint.cumulative"
```
~58% is null and overlaps with ```latest.admissions.admission_rate.overall``` and ```latest.school.open_admissions_policy```



In [5]:
tdf = collect()
# tdf = pd.read_csv(root/"data"/"clean"/"scorecard"/"clean_ml_scorecard_FL_programs.csv")
display(tdf.head())
display(tdf.info())

Total pages fetched: 3
Total Rows/Programs ingested: 2501


,code,title,unit_id,distance,school.type,credential.level,earnings.1_yr.overall_median_earnings,earnings.1_yr.working_not_enrolled.overall_count,earnings.4_yr.overall_median_earnings,earnings.4_yr.working_not_enrolled.overall_count,earnings.5_yr.overall_median_earnings,earnings.5_yr.working_not_enrolled.overall_count,id,school.name,location.lat,location.lon,latest.school.locale,latest.school.carnegie_size_setting,latest.admissions.admission_rate.overall,latest.student.demographics.median_family_income,latest.student.students_with_pell_grant,latest.school.open_admissions_policy,latest.student.demographics.age_entry,latest.school.title_iv.eligibility_type
0,1205,Culinary Arts and Related Services.,132374,1,Public,1,25586.0,22.0,22265,28,33286.0,21.0,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
1,4603,Electrical and Power Transmission Installers.,132374,2,Public,1,29493.0,32.0,41177,27,NaN,NaN,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
2,4702,"Heating, Air Conditioning, Ventilation and Ref...",132374,1,Public,1,36966.0,63.0,47325,48,NaN,NaN,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
3,4706,Vehicle Maintenance and Repair Technologies/Te...,132374,1,Public,1,34269.0,24.0,42839,28,NaN,NaN,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
4,5108,Allied Health and Medical Assisting Services.,132374,1,Public,1,32976.0,33.0,33081,29,NaN,NaN,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2501 entries, 0 to 2500
Data columns (total 24 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   code                                              2501 non-null   object 
 1   title                                             2501 non-null   object 
 2   unit_id                                           2501 non-null   int64  
 3   distance                                          2501 non-null   int64  
 4   school.type                                       2501 non-null   object 
 5   credential.level                                  2501 non-null   int64  
 6   earnings.1_yr.overall_median_earnings             2100 non-null   float64
 7   earnings.1_yr.working_not_enrolled.overall_count  2100 non-null   float64
 8   earnings.4_yr.overall_median_earnings             2501 non-null   int64  
 9   earnings.4_yr.worki

None

# Why so many missing Admission Rates?

In [6]:
df = tdf.copy()
round(df.isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.4226

In [7]:
round(df[df["latest.school.open_admissions_policy"]==2].isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.023

Approximately 40% of institutions have missing values for admission_rate.overall.
According to IPEDS reporting rules, institutions with an open admissions policy do not report traditional selectivity metrics such as admission rate or standardized test scores.

A check confirms that nearly all non-open-admission institutions report admission rates, indicating that the missingness is structural rather than random.

Therefore, missing admission rates are interpreted as corresponding primarily to open-admission institutions, and the open_admissions_policy variable is retained to preserve this structural distinction.

In [8]:
df = df.drop(columns="id")
df = clean(df)

Numeric columns: Index(['distance', 'credential_level', '1_yr_median_earnings',
       '1_yr_working_count', '4_yr_median_earnings', '4_yr_working_count',
       '5_yr_median_earnings', '5_yr_working_count', 'admission_rate_overall',
       'median_family_income', 'students_with_pell_grant'],
      dtype='object')


C:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\src\ira\clean\clean_scorecard.py:8: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df.columns = df.columns.str.replace(r".", "_")


In [9]:
df["selectivity_bucket"] = pd.cut(
    df["admission_rate_overall"],
    bins=[0, 0.3, 0.7, 1],
    labels=["elite", "mid", "open"]
)

In [10]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2501 entries, 0 to 2500
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   code                       2501 non-null   string  
 1   title                      2501 non-null   string  
 2   unit_id                    2501 non-null   string  
 3   distance                   2501 non-null   int64   
 4   school_type                2501 non-null   string  
 5   credential_level           2501 non-null   int64   
 6   1_yr_median_earnings       2100 non-null   float64 
 7   1_yr_working_count         2100 non-null   float64 
 8   4_yr_median_earnings       2501 non-null   int64   
 9   4_yr_working_count         2501 non-null   int64   
 10  5_yr_median_earnings       1788 non-null   float64 
 11  5_yr_working_count         1788 non-null   float64 
 12  school_name                2501 non-null   string  
 13  location_lat               2501 n

None

In [11]:
df.head()

,code,title,unit_id,distance,school_type,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,school_name,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket
0,1205,Culinary Arts and Related Services.,132374,1,Public,1,25586.0,22.0,22265,28,33286.0,21.0,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
1,4603,Electrical and Power Transmission Installers.,132374,2,Public,1,29493.0,32.0,41177,27,NaN,NaN,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
2,4702,"Heating, Air Conditioning, Ventilation and Ref...",132374,1,Public,1,36966.0,63.0,47325,48,NaN,NaN,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
3,4706,Vehicle Maintenance and Repair Technologies/Te...,132374,1,Public,1,34269.0,24.0,42839,28,NaN,NaN,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
4,5108,Allied Health and Medical Assisting Services.,132374,1,Public,1,32976.0,33.0,33081,29,NaN,NaN,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN


In [12]:
save(df,file_type="scorecard",clean=1,file_name="scorecard_FL_programs")
save(tdf,file_name="scorecard_FL_programs")

In [13]:
display("Relevant numeric variables statistics:", df.select_dtypes(exclude=object).describe())
display("Missing values per column:", df.isna().sum())
display("Correlation matrix:", df.corr(numeric_only=True))

'Relevant numeric variables statistics:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,admission_rate_overall,median_family_income,students_with_pell_grant
count,2501.000000,2501.000000,2100.000000,2100.000000,2501.000000,2501.000000,1788.000000,1788.000000,1444.000000,2491.000000,2303.000000
mean,1.431028,4.064774,47436.760952,135.764286,62386.475010,132.365054,60389.043624,140.090604,0.535130,29262.739864,0.734936
std,0.829784,10.392411,22519.842497,364.461526,25274.413965,382.210450,25962.542895,356.876270,0.228989,13060.807166,0.127915
min,0.000000,1.000000,11191.000000,16.000000,10611.000000,16.000000,12010.000000,16.000000,0.189000,0.000000,0.192053
25%,1.000000,2.000000,31025.000000,29.000000,46115.000000,26.000000,43159.750000,29.000000,0.401100,21349.000000,0.667383
50%,1.000000,3.000000,43529.500000,53.000000,57448.000000,46.000000,55379.000000,55.000000,0.546600,25118.000000,0.721905
75%,2.000000,3.000000,59504.250000,126.250000,75539.000000,111.000000,74449.250000,134.000000,0.696600,38662.000000,0.829670
max,3.000000,99.000000,246053.000000,10218.000000,221571.000000,9437.000000,271873.000000,6833.000000,1.000000,81806.000000,0.990401


'Missing values per column:'

code                            0
title                           0
unit_id                         0
distance                        0
school_type                     0
credential_level                0
1_yr_median_earnings          401
1_yr_working_count            401
4_yr_median_earnings            0
4_yr_working_count              0
5_yr_median_earnings          713
5_yr_working_count            713
school_name                     0
location_lat                    0
location_lon                    0
locale                          0
carnegie_size_setting           0
admission_rate_overall       1057
median_family_income           10
students_with_pell_grant      198
open_admissions_policy          4
age_entry                      10
title_iv_eligibility_type       0
selectivity_bucket           1057
dtype: int64

'Correlation matrix:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,admission_rate_overall,median_family_income,students_with_pell_grant
distance,1.000000,-0.153944,0.103553,0.056294,0.092568,0.073129,0.063412,0.074980,0.132306,-0.054622,0.056554
credential_level,-0.153944,1.000000,0.547171,-0.025134,0.048096,-0.029798,0.593513,-0.061411,-0.217481,0.005684,0.049736
1_yr_median_earnings,0.103553,0.547171,1.000000,0.046434,0.910971,0.012900,0.868921,-0.015934,-0.084154,0.196585,-0.180965
1_yr_working_count,0.056294,-0.025134,0.046434,1.000000,0.042067,0.934041,0.051325,0.774639,0.054556,-0.051221,0.026993
4_yr_median_earnings,0.092568,0.048096,0.910971,0.042067,1.000000,0.025095,0.935040,-0.013464,-0.215881,0.333229,-0.311550
4_yr_working_count,0.073129,-0.029798,0.012900,0.934041,0.025095,1.000000,0.026617,0.825968,0.058224,-0.059164,0.050633
5_yr_median_earnings,0.063412,0.593513,0.868921,0.051325,0.935040,0.026617,1.000000,-0.008875,-0.207934,0.332783,-0.320055
5_yr_working_count,0.074980,-0.061411,-0.015934,0.774639,-0.013464,0.825968,-0.008875,1.000000,0.056293,-0.091484,0.072359
admission_rate_overall,0.132306,-0.217481,-0.084154,0.054556,-0.215881,0.058224,-0.207934,0.056293,1.000000,-0.416269,0.393917
median_family_income,-0.054622,0.005684,0.196585,-0.051221,0.333229,-0.059164,0.332783,-0.091484,-0.416269,1.000000,-0.905237


In [14]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Reproducibility
RANDOM_STATE = 42

In [15]:
model_df = df.copy()
target = '4_yr_median_earnings'

drop_columns=["title","4_yr_working_count","school_name",'1_yr_median_earnings',
       '1_yr_working_count', '4_yr_median_earnings', '4_yr_working_count',
       '5_yr_median_earnings', '5_yr_working_count']
X = model_df.drop(columns=[target]).copy()
X = X.drop(columns=[],errors="ignore")
y = pd.to_numeric(model_df[target], errors='coerce').copy()
y_log = np.log10(y)

cat_cols = [
    'code',
    'school_type',
    'locale',
    'carnegie_size_setting',
    'open_admissions_policy',
    'title_iv_eligibility_type',
    'credential_level',
    'distance',
    "selectivity_bucket"
]
num_cols = [
    'admission_rate_overall',
    "location_lat",
    "location_lon", 
    'median_family_income',
    'students_with_pell_grant',
    'age_entry'
    
]

# categorical: force plain object and replace missing with np.nan
for c in cat_cols:
    X[c] = X[c].astype(object)
    X[c] = X[c].replace({pd.NA: np.nan})

# numeric: force numeric with np.nan
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

# nuclear option: remove any lingering pd.NA anywhere in X
X = X.astype(object).replace({pd.NA: np.nan})

# now restore numeric cols back to numeric dtype
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

In [59]:
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log,
    test_size=0.2,
    random_state=RANDOM_STATE
)

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
])

In [60]:
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', Ridge())
])

param_grid = {
    'reg__alpha': [0.01, 0.1, 1.0, 3.0, 10.0, 50.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid.fit(X_train, y_train_log)

print("Best params:", grid.best_params_)
print("Best CV MAE:", round(-grid.best_score_, 2))
best_model = grid.best_estimator_
preds = best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test_log, preds), 2))
print("Test R2:", round(r2_score(y_test_log, preds), 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'reg__alpha': 0.1}
Best CV MAE: 0.06
Test MAE: 0.06
Test R2: 0.761


In [61]:
ls_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Lasso(max_iter=100000, random_state=RANDOM_STATE))
])

ls_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0]
}

ls_grid = GridSearchCV(
    ls_pipe,
    param_grid=ls_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

ls_grid.fit(X_train, y_train_log)

print("Best params:", ls_grid.best_params_)
print("Best CV MAE:", round(-ls_grid.best_score_, 2))

ls_best_model = ls_grid.best_estimator_
ls_preds = ls_best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test_log, ls_preds), 2))
print("Test R2:", round(r2_score(y_test_log, ls_preds), 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'reg__alpha': 0.01}
Best CV MAE: 0.11
Test MAE: 0.11
Test R2: 0.3394


In [18]:
enet_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", ElasticNet(max_iter=100000))
])


enet_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0],
    "reg__l1_ratio": [0.005, 0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_grid = GridSearchCV(
    enet_pipe,
    param_grid=enet_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

enet_grid.fit(X_train, y_train_log)

print("ElasticNet Best params:", enet_grid.best_params_)
print("ElasticNet Best CV MAE:", round(-enet_grid.best_score_, 2))
enet_best_model = enet_grid.best_estimator_
enet_preds = enet_best_model.predict(X_test)

print("ElasticNet Test MAE:", round(mean_absolute_error(y_test_log, enet_preds), 2))
print("ElasticNet Test R2:", round(r2_score(y_test_log, enet_preds), 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
ElasticNet Best params: {'reg__alpha': 0.01, 'reg__l1_ratio': 0.005}
ElasticNet Best CV MAE: 0.08
ElasticNet Test MAE: 0.07
ElasticNet Test R2: 0.6744


In [19]:
baseline_pred = [y_train_log.mean()] * len(y_test_log)

print("Baseline MAE:", round(mean_absolute_error(y_test_log, baseline_pred), 2))

Baseline MAE: 0.13


In [29]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict


# pipe_rfr = Pipeline([
#     ('preprocessor', preprocessor),
#     ('clf', RandomForestRegressor(
#         random_state=RANDOM_STATE,
#         n_jobs=-1
#     ))
# ])

# param_grid_rfr = {
#     'clf__n_estimators': [100, 200, 300],
#     'clf__max_depth': [None, 5, 10, 20],
#     'clf__min_samples_split': [2, 5, 10],
#     'clf__min_samples_leaf': [1, 2, 4]
# }

# grid_rfr = GridSearchCV(
#     estimator=pipe_rfr,
#     param_grid=param_grid_rfr,
#     scoring='neg_mean_absolute_error',
#     cv=cv,
#     n_jobs=1,
#     refit=True,
#     verbose=1,
#     error_score='raise'
# )

# grid_rfr.fit(X_train, y_train_log)

# print("RF Best params:", grid_rfr.best_params_)
# print("RF Best CV MAE:", round(-grid_rfr.best_score_, 2))

# best_rfr = grid_rfr.best_estimator_
# rfr_preds = best_rfr.predict(X_test)

# print("RF Test MAE:", round(mean_absolute_error(y_test_log, rfr_preds), 2))
# print("RF Test R2:", round(r2_score(y_test_log, rfr_preds), 4))

In [30]:
from xgboost import XGBRegressor

# pipe_xgb = Pipeline([
#     ('preprocessor', preprocessor),
#     ('clf', XGBRegressor(
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#         verbosity=0
#     ))
# ])

# param_grid_xgb = {
#     'clf__n_estimators': [100, 200, 300],
#     'clf__max_depth': [None, 5, 10, 20],
#     'clf__learning_rate': [0.01, 0.05, 0.1],
#     'clf__subsample': [0.8, 1.0]
# }

# grid_xgb_log = GridSearchCV(
#     estimator=pipe_xgb,
#     param_grid=param_grid_xgb,
#     scoring='neg_mean_absolute_error',
#     cv=cv,
#     n_jobs=1,
#     refit=True,
#     verbose=1,
#     error_score='raise'
# )

# grid_xgb_log.fit(X_train, y_train_log)

# print("XGB Best params:", grid_xgb_log.best_params_)
# print("XGB Best CV MAE:", round(-grid_xgb_log.best_score_, 2))

# best_xgb_log = grid_xgb_log.best_estimator_
# xgb_preds_log = best_xgb_log.predict(X_test)

# print("XGB Test MAE:", round(mean_absolute_error(y_test_log, xgb_preds_log), 2))
# print("XGB Test R2:", round(r2_score(y_test_log, xgb_preds_log), 4))

Fitting 5 folds for each of 72 candidates, totalling 360 fits
XGB Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 10, 'clf__n_estimators': 300, 'clf__subsample': 0.8}
XGB Best CV MAE: 0.13
XGB Test MAE: 0.13
XGB Test R2: 0.7833

All models significantly outperformed the baseline. The linear model and Random Forest performed similarly, suggesting that linear relationships explain a large portion of the variance. However, XGBoost achieved the best performance, reducing MAE from 0.14 to 0.13 and increasing R2 to ~0.78. This indicates that nonlinear interactions exist in the data and are effectively captured by gradient boosting methods.

In [31]:
feature_names = best_xgb_log.named_steps["preprocessor"].get_feature_names_out()

importances = best_xgb_log.named_steps["clf"].feature_importances_

importance = pd.Series(importances, index=feature_names)

print("most important features")
display(importance.sort_values(ascending=False).head(15))
print("least important features")
display(importance.sort_values(ascending=True).head(15))

NameError: name 'best_xgb_log' is not defined

Feature importance analysis from the XGBoost model shows that categorical variables, particularly program codes (CIP), credential level, admission policy, and carnegie classification, were the most influential predictors. Additionally, several features had zero importance, indicating that certain categories did not contribute meaningfully to prediction, likely due to lack of signal.

The model was trained on log-transformed earnings to address skewness and improve predictive stability. Predictions were then exponentiated back to the original scale to evaluate performance and interpret errors in dollar terms.

Predictions are the typical (median-like) expected earnings rather than the average, which is appropriate given the skewed nature of income data.

In [32]:
def run_earnings_model(df, year, random_state=42, cv=5):
    year = int(year)
    target = f"{year}_yr_median_earnings"

    # columns from other horizons to drop
    other_year_cols = []
    for y in [1, 4, 5]:
        if y != year:
            other_year_cols.extend([
                f"{y}_yr_median_earnings",
                f"{y}_yr_working_count",
            ])

    model_df = df.copy()

    # clean target
    model_df[target] = pd.to_numeric(model_df[target], errors="coerce")
    model_df = model_df[model_df[target].notna() & (model_df[target] > 0)].copy()

    # build X / y
    drop_columns = ["title", "school_name"] + other_year_cols

    X = model_df.drop(columns=drop_columns, errors="ignore").copy()
    X = X.drop(columns=[target], errors="ignore")
    y = pd.to_numeric(model_df[target], errors="coerce").copy()
    y_log = np.log(y)

    

    cat_cols = [
    'code',
    'school_type',
    'locale',
    'carnegie_size_setting',
    'open_admissions_policy',
    'title_iv_eligibility_type',
    'credential_level',
    'distance',
    "selectivity_bucket"
    ]
    
    num_cols = [
        'admission_rate_overall',
        "location_lat",
        "location_lon", 
        'median_family_income',
        'students_with_pell_grant',
        'age_entry'
    ]

    cat_cols = [c for c in cat_cols if c in X.columns]
    num_cols = [c for c in num_cols if c in X.columns]

    # coerce dtypes
    for c in cat_cols:
        X[c] = X[c].astype(object).replace({pd.NA: np.nan})

    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    X = X.astype(object).replace({pd.NA: np.nan})

    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # split
    X_train, X_test, y_train_log, y_test_log = train_test_split(
        X, y_log, test_size=0.2, random_state=random_state
    )

    # preprocessing
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(missing_values=np.nan, strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(missing_values=np.nan, strategy="median")),
        ("scaler", StandardScaler()),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ])

    # model
    pipe_xgb = Pipeline([
        ("preprocessor", preprocessor),
        ("reg", XGBRegressor(
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )),
    ])

    param_grid_xgb = {
        "reg__n_estimators": [100, 200, 300],
        "reg__max_depth": [3, 5, 10],
        "reg__learning_rate": [0.01, 0.05, 0.1],
        "reg__subsample": [0.8, 1.0],
    }

    grid_xgb_log = GridSearchCV(
        estimator=pipe_xgb,
        param_grid=param_grid_xgb,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=1,
        refit=True,
        verbose=1,
        error_score="raise",
    )

    # fit
    grid_xgb_log.fit(X_train, y_train_log)

    print(f"\n===== {year}-YEAR MODEL =====")
    print("XGB Best params:", grid_xgb_log.best_params_)
    print("XGB Best CV MAE (log):", round(-grid_xgb_log.best_score_, 4))

    best_xgb_log = grid_xgb_log.best_estimator_
    xgb_preds_log = best_xgb_log.predict(X_test)

    print("XGB Test MAE (log):", round(mean_absolute_error(y_test_log, xgb_preds_log), 4))
    print("XGB Test R2 (log):", round(r2_score(y_test_log, xgb_preds_log), 4))

    # cross-validated preds on all rows
    cv_preds_log = cross_val_predict(
        best_xgb_log,
        X,
        y_log,
        cv=cv,
        method="predict",
        n_jobs=1,
    )

    # dynamic column names
    actual_log_col = f"{year}_year_earning_log"
    pred_log_col = f"{year}_year_pred_log"
    actual_col = f"{year}_year_earning"
    pred_col = f"{year}_year_pred"
    error_col = f"{year}_year_error"

    error_df = X.copy()
    error_df[actual_log_col] = y_log
    error_df[pred_log_col] = cv_preds_log

    # add metadata back
    cols_to_add = [c for c in ["title", "school_name", "credential_level"] if c in model_df.columns]
    error_df[cols_to_add] = model_df.loc[error_df.index, cols_to_add]

    # delog dynamically
    delog_col_map = {
        actual_log_col: actual_col,
        pred_log_col: pred_col,
    }

    for log_col, out_col in delog_col_map.items():
        error_df[out_col] = np.exp(error_df[log_col])

    error_df[error_col] = error_df[actual_col] - error_df[pred_col]

    return {
        "year": year,
        "target": target,
        "grid": grid_xgb_log,
        "best_model": best_xgb_log,
        "X": X,
        "y": y,
        "error_df": error_df,
    }

In [33]:
results_1 = run_earnings_model(df, year=1, random_state=RANDOM_STATE, cv=cv)
results_4 = run_earnings_model(df, year=4, random_state=RANDOM_STATE, cv=cv)
results_5 = run_earnings_model(df, year=5, random_state=RANDOM_STATE, cv=cv)

Fitting 5 folds for each of 54 candidates, totalling 270 fits

===== 1-YEAR MODEL =====
XGB Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
XGB Best CV MAE (log): 0.1515
XGB Test MAE (log): 0.1258
XGB Test R2 (log): 0.8581
Fitting 5 folds for each of 54 candidates, totalling 270 fits

===== 4-YEAR MODEL =====
XGB Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
XGB Best CV MAE (log): 0.1323
XGB Test MAE (log): 0.1278
XGB Test R2 (log): 0.7895
Fitting 5 folds for each of 54 candidates, totalling 270 fits

===== 5-YEAR MODEL =====
XGB Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
XGB Best CV MAE (log): 0.1387
XGB Test MAE (log): 0.12
XGB Test R2 (log): 0.8615


In [34]:
df_1=results_1["error_df"]
display(df_1.head())
display(results_1["error_df"].shape)

df_4=results_4["error_df"]
display(df_4.head())
display(results_4["error_df"].shape)

df_5=results_5["error_df"]
display(df_5.head())
display(results_5["error_df"].shape)

,code,unit_id,distance,school_type,credential_level,1_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,1_year_earning_log,1_year_pred_log,title,school_name,1_year_earning,1_year_pred,1_year_error
0,1205,132374,1,Public,1,22.0,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.149801,9.877033,Culinary Arts and Related Services.,Atlantic Technical College,25586.0,19477.851562,6108.148437
1,4603,132374,2,Public,1,32.0,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.291908,10.367987,Electrical and Power Transmission Installers.,Atlantic Technical College,29493.0,31824.339844,-2331.339844
2,4702,132374,1,Public,1,63.0,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.517754,10.527230,"Heating, Air Conditioning, Ventilation and Ref...",Atlantic Technical College,36966.0,37317.968750,-351.968750
3,4706,132374,1,Public,1,24.0,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.441996,10.357135,Vehicle Maintenance and Repair Technologies/Te...,Atlantic Technical College,34269.0,31480.853516,2788.146484
4,5108,132374,1,Public,1,33.0,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.403535,10.278213,Allied Health and Medical Assisting Services.,Atlantic Technical College,32976.0,29091.826172,3884.173828


(2100, 24)

,code,unit_id,distance,school_type,credential_level,4_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,4_year_earning_log,4_year_pred_log,title,school_name,4_year_earning,4_year_pred,4_year_error
0,1205,132374,1,Public,1,28,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.010771,10.304316,Culinary Arts and Related Services.,Atlantic Technical College,22265.0,29861.208984,-7596.208984
1,4603,132374,2,Public,1,27,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.625635,10.528167,Electrical and Power Transmission Installers.,Atlantic Technical College,41177.0,37352.937500,3824.062500
2,4702,132374,1,Public,1,48,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.764794,10.566911,"Heating, Air Conditioning, Ventilation and Ref...",Atlantic Technical College,47325.0,38828.535156,8496.464844
3,4706,132374,1,Public,1,28,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.665204,10.841309,Vehicle Maintenance and Repair Technologies/Te...,Atlantic Technical College,42839.0,51088.187500,-8249.187500
4,5108,132374,1,Public,1,29,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.406714,10.442006,Allied Health and Medical Assisting Services.,Atlantic Technical College,33081.0,34269.332031,-1188.332031


(2501, 24)

,code,unit_id,distance,school_type,credential_level,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error
0,1205,132374,1,Public,1,21.0,26.242780,-80.192271,21,-2,NaN,16748.0,NaN,1,26.0,1,NaN,10.412892,9.888281,Culinary Arts and Related Services.,Atlantic Technical College,33286.0,19698.167969,13587.832031
8,1101,132471,2,"Private, nonprofit",3,68.0,25.878908,-80.198925,21,13,0.7724,28822.0,0.712681,2,26.0,1,open,11.029520,11.313852,"Computer and Information Sciences, General.",Barry University,61668.0,81948.992188,-20280.992188
9,1304,132471,2,"Private, nonprofit",5,86.0,25.878908,-80.198925,21,13,0.7724,28822.0,0.712681,2,26.0,1,open,11.266628,11.215052,Educational Administration and Supervision.,Barry University,78169.0,74239.500000,3929.500000
10,1313,132471,2,"Private, nonprofit",5,46.0,25.878908,-80.198925,21,13,0.7724,28822.0,0.712681,2,26.0,1,open,11.119542,11.085058,Teacher Education and Professional Development...,Barry University,67477.0,65189.792969,2287.207031
11,2201,132471,1,"Private, nonprofit",7,204.0,25.878908,-80.198925,21,13,0.7724,28822.0,0.712681,2,26.0,1,open,11.262154,11.445946,Law.,Barry University,77820.0,93521.406250,-15701.406250


(1788, 24)

In [35]:
df_1['1_yr_working_count'].isna().sum()

0

In [36]:
merge_df=df_5.merge(df_4[['4_year_earning','4_year_earning_log','4_year_pred','4_year_pred_log','4_year_error',"4_yr_working_count",'code','unit_id','credential_level']],on=['code','unit_id','credential_level'],how="inner")
merge_df=merge_df.merge(df_1[['1_year_earning','1_year_earning_log','1_year_pred','1_year_pred_log','1_year_error',"1_yr_working_count",'code','unit_id','credential_level']],on=['code','unit_id','credential_level'],how="inner")
merge_df.shape

(1610, 36)

In [37]:
error_df=merge_df.copy()
valid_codes = (
    error_df.groupby(["code","credential_level"])["school_name"]
    .nunique()
)

valid_codes = valid_codes[valid_codes >= 3].index
valid_codes

MultiIndex([('0301', 3),
            ('0402', 3),
            ('0402', 5),
            ('0901', 3),
            ('0904', 3),
            ('0907', 3),
            ('0909', 3),
            ('1101', 3),
            ('1101', 5),
            ('1104', 3),
            ...
            ('5208', 5),
            ('5209', 3),
            ('5210', 3),
            ('5210', 5),
            ('5211', 3),
            ('5212', 3),
            ('5212', 5),
            ('5213', 3),
            ('5214', 3),
            ('5401', 3)],
           names=['code', 'credential_level'], length=147)

In [38]:
# error_df=error_df[error_df["4_yr_working_count"] >= 20]


error_df = (
    error_df
    .set_index(["code", "credential_level"])
    .loc[valid_codes]
    .reset_index()
)

error_df["school_count"] = (
    error_df.groupby(["code", "credential_level"])["school_name"]
    .transform("nunique")
)

error_df["confidence"] = pd.cut(
    error_df["school_count"],
    bins=[0, 5, 15, 100],
    labels=["low", "medium", "high"]
)

In [39]:
error_df.shape

(1367, 38)

In [40]:
years = ["1", "4", "5"]

for y in years:
    # percent error
    error_df[f"{y}_year_pct_error"] = (
        error_df[f"{y}_year_error"] / error_df[f"{y}_year_pred"]
    )

    # weight (you can keep using 4yr count OR match per year if you have it)
    k = np.percentile(np.log1p(error_df[f"{y}_yr_working_count"]), 75)

    weight = (
        np.log1p(error_df[f"{y}_yr_working_count"]) /
        np.log1p(error_df[f"{y}_yr_working_count"] + k)
    )

    # final score per year
    error_df[f"{y}_year_score"] = (
        error_df[f"{y}_year_pct_error"] * weight
    )

“The weighting scheme introduces only minor adjustments to ranking positions, suggesting that prediction error remains dominant while program size provides a secondary refinement.”

To avoid instability in groups with small sample sizes, a small constant (epsilon) was added to the standard deviation when computing the final score. This prevents artificially inflated scores caused by near-zero variance estimates, while still allowing all groups to be included in the analysis.

To account for differences in sample size, a soft penalization factor was applied using sqrt(n / (n + k)). This approach reduces the influence of groups with small sample sizes without excluding them entirely. As n increases, the penalty diminishes, allowing larger groups to retain their full weight while appropriately down-weighting less reliable estimates.

In [41]:
error_df["rank_1"] = error_df.groupby(["code","credential_level"])["1_year_score"].rank(ascending=False, method="min")
error_df["rank_4"] = error_df.groupby(["code","credential_level"])["4_year_score"].rank(ascending=False, method="min")
error_df["rank_5"] = error_df.groupby(["code","credential_level"])["5_year_score"].rank(ascending=False, method="min")

In [42]:
error_df["move_1_to_4"] = error_df["rank_4"] - error_df["rank_1"]
error_df["move_4_to_5"] = error_df["rank_5"] - error_df["rank_4"]
error_df["move_1_to_5"] = error_df["rank_5"] - error_df["rank_1"]

In [43]:
error_df["rank_std"] = error_df[["rank_1","rank_4","rank_5"]].std(axis=1)
group_size = error_df.groupby("code")["code"].transform("count")

error_df["rank_std_pct"] = error_df["rank_std"] / (group_size - 1)

In [44]:
error_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct
0,0301,3,133492,1,"Private, nonprofit",56.0,27.715798,-82.687043,11,11,0.7580,77695.0,0.411392,2,21.0,1,open,10.662914,10.643175,Natural Resources Conservation and Research.,Eckerd College,42741.0,41905.617188,835.382813,39932.0,10.594933,50566.960938,10.831054,-10634.960938,77,25709.0,10.154596,29107.587891,10.278754,-3398.587891,94.0,8,medium,-0.116760,-0.115420,-0.210314,-0.207225,0.019935,0.019527,6.0,8.0,3.0,2.0,-5.0,-3.0,2.516611,0.359516
1,0301,3,133951,2,Public,36.0,25.757320,-80.373928,21,15,0.5466,23924.0,0.813127,2,23.0,1,mid,10.745076,10.783957,Natural Resources Conservation and Research.,Florida International University,46401.0,48240.609375,-1839.609375,49748.0,10.814726,55158.484375,10.917966,-5410.484375,24,36007.0,10.491469,33721.535156,10.425892,2285.464844,21.0,8,medium,0.067775,0.063452,-0.098090,-0.092620,-0.038134,-0.036835,2.0,7.0,6.0,5.0,-1.0,4.0,2.645751,0.377964
2,0301,3,134097,1,Public,78.0,30.443147,-84.295064,12,15,0.2422,43729.0,0.597532,2,20.0,1,elite,10.780705,10.786012,Natural Resources Conservation and Research.,Florida State University,48084.0,48339.851562,-255.851562,54288.0,10.902058,52498.199219,10.868534,1789.800781,188,30146.0,10.313808,36875.710938,10.515308,-6729.710937,161.0,8,medium,-0.182497,-0.181380,0.034093,0.033916,-0.005293,-0.005219,7.0,5.0,5.0,-2.0,0.0,-2.0,1.154701,0.164957
3,0301,3,134130,1,Public,28.0,29.646290,-82.347911,12,16,0.2420,39127.0,0.667383,2,21.0,1,elite,10.895108,10.878085,Natural Resources Conservation and Research.,University of Florida,53912.0,53002.011719,909.988281,62683.0,11.045846,57587.316406,10.961058,5095.683594,29,34454.0,10.447380,36076.906250,10.493408,-1622.906250,29.0,8,medium,-0.044985,-0.042980,0.088486,0.084491,0.017169,0.016391,5.0,3.0,4.0,-2.0,1.0,-1.0,1.000000,0.142857
4,0301,3,136950,1,"Private, nonprofit",29.0,28.592787,-81.349239,21,11,0.4754,43978.0,0.580000,2,22.0,1,mid,10.517050,10.832608,Natural Resources Conservation and Research.,Rollins College,36940.0,50645.628906,-13705.628906,63363.0,11.056635,52899.144531,10.876143,10463.855469,23,22352.0,10.014671,34313.710938,10.443300,-11961.710937,18.0,8,medium,-0.348599,-0.322316,0.197808,0.186247,-0.270618,-0.258835,8.0,2.0,8.0,-6.0,6.0,0.0,3.464102,0.494872


In [45]:
error_df.shape

(1367, 52)

In [46]:
error_df["rank_std_pct"].describe()

count    1367.000000
mean        0.159253
std         0.132020
min         0.000000
25%         0.054986
50%         0.126053
75%         0.231118
max         0.577350
Name: rank_std_pct, dtype: float64

When ranking schools within the same program, we observe an average rank movement of about 12%, indicating that performance is not stable over time even within comparable fields.

In [47]:
import plotly.express as px

px.histogram(error_df["rank_std_pct"])

In [48]:
error_df[["rank_std_pct", "4_yr_working_count"]].corr()

,rank_std_pct,4_yr_working_count
rank_std_pct,1.000000,-0.084436
4_yr_working_count,-0.084436,1.000000


In [49]:
top = error_df.nsmallest(100, "rank_4")   # best schools
bottom = error_df.nlargest(100, "rank_4")  # worst schools

print('top 100 std:',top["rank_std_pct"].mean())
print('bottom 100 std:',bottom["rank_std_pct"].mean())

top 100 std: 0.2446378002835811
bottom 100 std: 0.15649586161136464


We tested whether ranking instability was due to small sample sizes, but found no meaningful relationship. Even top-performing schools show similar or increased volatility, suggesting the instability is inherent to the earnings metric itself rather than noise.

In [50]:
program_stability = (
    error_df
    .groupby(["code", "credential_level"])
    .agg(
        mean_rank_std_pct=("rank_std_pct", "mean"),
        n=("school_name", "nunique")
    )
    .reset_index()
)

stable_programs = program_stability[
    program_stability["mean_rank_std_pct"] < 2
]
stable_df = error_df.merge(
    stable_programs[["code", "credential_level",'mean_rank_std_pct']],
    on=["code", "credential_level"],
    how="inner"
)
len(stable_df) / len(error_df)

1.0

In [51]:
stable_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct
0,0301,3,133492,1,"Private, nonprofit",56.0,27.715798,-82.687043,11,11,0.7580,77695.0,0.411392,2,21.0,1,open,10.662914,10.643175,Natural Resources Conservation and Research.,Eckerd College,42741.0,41905.617188,835.382813,39932.0,10.594933,50566.960938,10.831054,-10634.960938,77,25709.0,10.154596,29107.587891,10.278754,-3398.587891,94.0,8,medium,-0.116760,-0.115420,-0.210314,-0.207225,0.019935,0.019527,6.0,8.0,3.0,2.0,-5.0,-3.0,2.516611,0.359516,0.250313
1,0301,3,133951,2,Public,36.0,25.757320,-80.373928,21,15,0.5466,23924.0,0.813127,2,23.0,1,mid,10.745076,10.783957,Natural Resources Conservation and Research.,Florida International University,46401.0,48240.609375,-1839.609375,49748.0,10.814726,55158.484375,10.917966,-5410.484375,24,36007.0,10.491469,33721.535156,10.425892,2285.464844,21.0,8,medium,0.067775,0.063452,-0.098090,-0.092620,-0.038134,-0.036835,2.0,7.0,6.0,5.0,-1.0,4.0,2.645751,0.377964,0.250313
2,0301,3,134097,1,Public,78.0,30.443147,-84.295064,12,15,0.2422,43729.0,0.597532,2,20.0,1,elite,10.780705,10.786012,Natural Resources Conservation and Research.,Florida State University,48084.0,48339.851562,-255.851562,54288.0,10.902058,52498.199219,10.868534,1789.800781,188,30146.0,10.313808,36875.710938,10.515308,-6729.710937,161.0,8,medium,-0.182497,-0.181380,0.034093,0.033916,-0.005293,-0.005219,7.0,5.0,5.0,-2.0,0.0,-2.0,1.154701,0.164957,0.250313
3,0301,3,134130,1,Public,28.0,29.646290,-82.347911,12,16,0.2420,39127.0,0.667383,2,21.0,1,elite,10.895108,10.878085,Natural Resources Conservation and Research.,University of Florida,53912.0,53002.011719,909.988281,62683.0,11.045846,57587.316406,10.961058,5095.683594,29,34454.0,10.447380,36076.906250,10.493408,-1622.906250,29.0,8,medium,-0.044985,-0.042980,0.088486,0.084491,0.017169,0.016391,5.0,3.0,4.0,-2.0,1.0,-1.0,1.000000,0.142857,0.250313
4,0301,3,136950,1,"Private, nonprofit",29.0,28.592787,-81.349239,21,11,0.4754,43978.0,0.580000,2,22.0,1,mid,10.517050,10.832608,Natural Resources Conservation and Research.,Rollins College,36940.0,50645.628906,-13705.628906,63363.0,11.056635,52899.144531,10.876143,10463.855469,23,22352.0,10.014671,34313.710938,10.443300,-11961.710937,18.0,8,medium,-0.348599,-0.322316,0.197808,0.186247,-0.270618,-0.258835,8.0,2.0,8.0,-6.0,6.0,0.0,3.464102,0.494872,0.250313


In [52]:
stable_programs.shape

(147, 4)

In [53]:
stable_gap_df = stable_df.copy()

stable_gap_df["median_score"] = stable_gap_df[
    ["1_year_score", "4_year_score", "5_year_score"]
].median(axis=1)

# program-level gap stats
program_gap = (
    stable_gap_df
    .groupby(["code", "credential_level"])
    .agg(
        min_score=("median_score", "min"),
        q1_score=("median_score", lambda x: x.quantile(0.25)),
        median_program_score=("median_score", "median"),
        q3_score=("median_score", lambda x: x.quantile(0.75)),
        max_score=("median_score", "max"),
        mean_score=("median_score", "mean"),
        std_score=("median_score", "std"),
        n=("school_name", "nunique")
    )
    .reset_index()
)

# raw gap: biggest under/over performer spread 
program_gap["gap"] = program_gap["max_score"] - program_gap["min_score"]

# robust gap: less sensitive to one weird school 
program_gap["iqr_gap"] = program_gap["q3_score"] - program_gap["q1_score"]

#  weight so tiny groups don't dominate 
program_gap["gap_weighted"] = (
    program_gap["gap"] * (program_gap["n"] / (program_gap["n"] + 10))
)
program_gap["iqr_gap_weighted"] = (
    program_gap["iqr_gap"] * (program_gap["n"] / (program_gap["n"] + 10))
)

#  bring back titles 
titles = error_df[["code", "credential_level", "title"]].drop_duplicates()

program_gap = program_gap.merge(
    titles,
    on=["code", "credential_level"],
    how="left"
)

# top programs with biggest stable-school spread 
top_gap_programs = (
    program_gap
    .sort_values("gap_weighted", ascending=False)
)

top_iqr_programs = (
    program_gap
    .sort_values("iqr_gap_weighted", ascending=False)
)

# views
top_gap_programs.head(5)

,code,credential_level,min_score,q1_score,median_program_score,q3_score,max_score,mean_score,std_score,n,gap,iqr_gap,gap_weighted,iqr_gap_weighted,title
125,5138,2,-0.569435,-0.023707,0.020034,0.050910,0.191705,0.007942,0.107819,50,0.761140,0.074617,0.634283,0.062181,"Registered Nursing, Nursing Administration, Nu..."
127,5138,5,-0.148095,-0.092409,-0.042823,0.028642,0.884964,0.038022,0.261766,14,1.033059,0.121051,0.602617,0.070613,"Registered Nursing, Nursing Administration, Nu..."
128,5139,1,-0.363771,-0.028030,0.042909,0.088551,0.356672,0.039672,0.128142,35,0.720443,0.116581,0.560345,0.090674,"Practical Nursing, Vocational Nursing and Nurs..."
7,1101,3,-0.308869,-0.053512,0.033830,0.069359,0.438867,0.020570,0.151661,19,0.747736,0.122871,0.489896,0.080502,"Computer and Information Sciences, General."
113,5109,2,-0.233298,-0.080442,0.032319,0.091740,0.297263,0.024873,0.125775,33,0.530561,0.172182,0.407175,0.132140,"Allied Health Diagnostic, Intervention, and Tr..."


In [54]:
top_iqr_programs.head(5)

,code,credential_level,min_score,q1_score,median_program_score,q3_score,max_score,mean_score,std_score,n,gap,iqr_gap,gap_weighted,iqr_gap_weighted,title
89,4706,1,-0.224808,-0.152770,-0.023376,0.158410,0.384711,0.019625,0.196791,15,0.609519,0.311179,0.365711,0.186707,Vehicle Maintenance and Repair Technologies/Te...
68,4301,1,-0.115880,-0.028859,0.064254,0.194649,0.343399,0.087810,0.149409,17,0.459279,0.223508,0.289175,0.140727,Criminal Justice and Corrections.
132,5202,5,-0.164374,-0.052337,0.004833,0.123802,0.331209,0.047163,0.126367,33,0.495583,0.176139,0.380331,0.135176,"Business Administration, Management and Operat..."
113,5109,2,-0.233298,-0.080442,0.032319,0.091740,0.297263,0.024873,0.125775,33,0.530561,0.172182,0.407175,0.132140,"Allied Health Diagnostic, Intervention, and Tr..."
76,4404,3,-0.058147,-0.044825,0.007896,0.294241,0.520627,0.134260,0.250273,6,0.578774,0.339065,0.217040,0.127150,Public Administration.


In [55]:
print(
    program_stability[
        (program_stability["code"] == "4301") &
        (program_stability["credential_level"] == 1)
    ]
)

    code  credential_level  mean_rank_std_pct   n
68  4301                 1           0.050242  17


In [56]:
(
    stable_gap_df[(stable_gap_df["code"] == "4301")&(stable_gap_df["credential_level"]==1)]
    .sort_values(["credential_level", "median_score"], ascending=[True, False])
)

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score
520,4301,1,136358,1,Public,139.0,26.612560,-80.086530,21,15,NaN,21408.0,0.921804,1,23.0,1,NaN,11.288531,10.756253,Criminal Justice and Corrections.,Palm Beach State College,79900.0,46922.527344,32977.472656,78216.0,11.267230,58123.898438,10.970332,20092.101562,153,69415.0,11.147858,52569.238281,10.869886,16845.761719,168.0,17,high,0.320449,0.318583,0.345677,0.343399,0.702807,0.697828,2.0,1.0,1.0,-1.0,0.0,-1.0,0.577350,0.010893,0.050242,0.343399
511,4301,1,133702,1,Public,34.0,30.334918,-81.659901,11,15,NaN,25081.0,0.815958,1,25.0,1,NaN,11.047312,10.777512,Criminal Justice and Corrections.,Florida State College at Jacksonville,62775.0,47930.703125,14844.296875,66209.0,11.100572,56887.152344,10.948825,9321.847656,64,60055.0,11.003016,45981.539062,10.735995,14073.460938,105.0,17,high,0.306068,0.302982,0.163866,0.160881,0.309703,0.298444,3.0,6.0,4.0,3.0,-2.0,1.0,1.527525,0.028821,0.050242,0.298444
512,4301,1,134291,1,Public,21.0,30.487547,-87.290659,21,-2,NaN,15300.0,NaN,1,29.0,1,NaN,10.920709,10.607920,Criminal Justice and Corrections.,George Stone Technical College,55310.0,40453.957031,14856.042969,53477.0,10.887007,41090.972656,10.623544,12386.027344,22,45951.0,10.735331,37758.761719,10.538973,8192.238281,31.0,17,high,0.216963,0.207997,0.301429,0.282929,0.367233,0.344322,5.0,2.0,3.0,-3.0,1.0,-2.0,1.527525,0.028821,0.050242,0.282929
523,4301,1,138497,1,Public,22.0,28.834861,-82.345563,23,-2,NaN,17183.0,NaN,1,28.0,1,NaN,10.856862,10.611962,Criminal Justice and Corrections.,Withlacoochee Technical College,51889.0,40617.828125,11271.171875,52652.0,10.871460,41125.156250,10.624375,11526.843750,29,37788.0,10.539747,40078.550781,10.598597,-2290.550781,18.0,17,high,-0.057152,-0.052843,0.280287,0.267633,0.277493,0.261045,14.0,3.0,5.0,-11.0,2.0,-9.0,5.859465,0.110556,0.050242,0.261045
515,4301,1,134608,1,Public,48.0,27.424160,-80.358682,21,12,NaN,21577.0,0.912263,1,23.0,1,NaN,11.088216,10.716019,Criminal Justice and Corrections.,Indian River State College,65396.0,45072.101562,20323.898438,69613.0,11.150707,58109.980469,10.970093,11503.019531,69,60298.0,11.007054,52909.792969,10.876344,7388.207031,51.0,17,high,0.139638,0.136375,0.197953,0.194649,0.450920,0.439887,8.0,4.0,2.0,-4.0,-2.0,-6.0,3.055050,0.057642,0.050242,0.194649
517,4301,1,135267,1,Public,46.0,26.647517,-81.836596,13,-2,NaN,19016.0,NaN,1,26.0,1,NaN,10.878104,10.782584,Criminal Justice and Corrections.,Fort Myers Technical College,53003.0,48174.453125,4828.546875,73379.0,11.203393,61804.566406,11.031733,11574.433594,28,58949.0,10.984428,49547.070312,10.810678,9401.929688,17.0,17,high,0.189758,0.174546,0.187275,0.178480,0.100230,0.097655,6.0,5.0,8.0,-1.0,3.0,2.0,1.527525,0.028821,0.050242,0.174546
514,4301,1,134495,1,Public,151.0,27.934888,-82.456253,11,5,NaN,22087.0,0.830344,1,24.0,1,NaN,11.035099,11.020070,Criminal Justice and Corrections.,Hillsborough Community College,62013.0,61087.957031,925.042969,66994.0,11.112358,58886.992188,10.983376,8107.007813,238,62249.0,11.038898,51157.808594,10.842670,11091.191406,225.0,17,high,0.216803,0.215905,0.137671,0.137128,0.015143,0.015045,4.0,7.0,11.0,3.0,4.0,7.0,3.511885,0.066262,0.050242,0.137128
510,4301,1,133386,1,Public,147.0,29.202780,-81.049513,13,12,NaN,21285.0,0.853

In [57]:
import plotly.express as px
bar_df = stable_gap_df[
    (stable_gap_df["code"] == "4301") & (stable_gap_df["credential_level"] == 1)
]

bar_df = bar_df.sort_values(by="median_score", ascending=True)

px.bar(
    bar_df,
    x="median_score",
    y="school_name",
    orientation="h",
    labels={"median_score":"3 year Pred. Error Average","school_name":"Schools"},
    title="Certificate in Criminal Justice and Corrections."
)

Programs such as Public Health show substantial variability in outcomes across institutions, even after controlling for observable factors. This suggests that institutional effects such as program quality, networking opportunities, or industry connections may play a significant role in shaping student outcomes.

In [58]:
save(stable_gap_df,file_name="residual_FL_stable_programs")